# 商品销售数据分析

## 数字素养 - 综合项目实践

**姓名：张敖博**

---

本项目使用 Python 对某电商平台的商品销售数据进行多维度的数据分析，包括月度销售额统计、商品销售趋势分析、品类对比分析等，为电商运营和营销策略提供数据支持。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# 设置中文字体，避免图表中文显示乱码
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print('库导入成功！')

## 1. 读取数据

使用 `pandas.read_excel()` 函数分别读取工作簿中的两个工作表：
- **信息表**：包含产品基本信息（产品大类、小类、名称、编号、售价）
- **销售数据表**：包含销售记录（销售日期、销售单号、产品号、销售数量）

In [ ]:
# 读取两个工作表
df_info = pd.read_excel('商品销售数据.xlsx', sheet_name=0)  # 信息表
df_sales = pd.read_excel('商品销售数据.xlsx', sheet_name=1)  # 销售数据表

print('=== 信息表（前5行）===')
display(df_info.head())
print(f'\n信息表共有 {len(df_info)} 条产品记录')
print(f'列名：{list(df_info.columns)}')

print('\n=== 销售数据表（前5行）===')
display(df_sales.head())
print(f'\n销售数据表共有 {len(df_sales)} 条销售记录')
print(f'列名：{list(df_sales.columns)}')

## 2. 合并数据

使用 `pandas.merge()` 函数按照 **产品号** 将信息表和销售数据表合并，形成完整的销售明细数据。

In [ ]:
# 按产品号合并两个表
df = pd.merge(df_sales, df_info, on='产品号', how='inner')
print(f'合并后共有 {len(df)} 条记录')
print(f'\n合并后列名：{list(df.columns)}')
print('\n=== 合并后数据（前10行）===')
display(df.head(10))
print('\n数据缺失检查：')
print(df.isnull().sum())

## 3. 数据统计

### 计算每月销售金额

- **销售金额** = 产品售价 × 销售数量
- 从销售日期中提取 **月份** 信息（使用 `dt.month`）
- 按月份使用 `groupby()` + `sum()` 进行分组汇总

In [ ]:
# 计算销售金额（新增一列）
df['销售金额'] = df['产品售价'] * df['销售数量']

# 从销售日期中提取月份
df['月份'] = df['销售日期'].dt.month

print('=== 添加计算列后的数据（前5行）===')
display(df[['销售日期', '月份', '产品号', '产品名称', '销售数量', '产品售价', '销售金额']].head())

# 按月份统计销售总额
monthly_sales = df.groupby('月份')['销售金额'].sum().reset_index()
monthly_sales.columns = ['月份', '销售总额']
print('\n=== 每月销售总额 ===')
display(monthly_sales)
print(f'\n全年销售总额：{monthly_sales["销售总额"].sum():,.2f} 元')

# 月度增长率
monthly_sales['环比增长率(%)'] = monthly_sales['销售总额'].pct_change() * 100
print('\n=== 含环比增长率的月度销售数据 ===')
display(monthly_sales)

## 4. 商品每月销售变化趋势

提取 **月份** 和 **销售金额** 列，使用 `plot()` 函数绘制折线图，展示各产品每月的销售金额变化趋势。
- X轴：月份
- Y轴：销售金额

In [ ]:
# 按月份和产品号分组汇总销售金额
product_monthly = df.groupby(['月份', '产品号', '产品名称'])['销售金额'].sum().reset_index()

# 选择销售总额最高的前10个产品展示趋势
top10_products = df.groupby('产品号')['销售金额'].sum().nlargest(10).index
df_top10 = product_monthly[product_monthly['产品号'].isin(top10_products)]

# 绘制折线图
fig, ax = plt.subplots(figsize=(14, 8))

# 使用不同颜色和标记
colors = plt.cm.tab10(range(10))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

for i, pid in enumerate(top10_products):
    data = df_top10[df_top10['产品号'] == pid].sort_values('月份')
    product_name = data['产品名称'].iloc[0]
    ax.plot(data['月份'], data['销售金额'], marker=markers[i], linewidth=2,
            color=colors[i], label=f'{pid} {product_name}', markersize=6)

ax.set_xlabel('月份', fontsize=12)
ax.set_ylabel('销售金额（元）', fontsize=12)
ax.set_title('销售额TOP10产品 - 每月销售趋势', fontsize=16, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels([f'{m}月' for m in range(1, 13)])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('product_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存为 product_monthly_trend.png')

## 5. 商品大类每月销售变化对比

按照 **月份** 和 **产品大类** 分组汇总销售金额，使用 `bar()` 绘制柱状图，对比不同产品大类在各月的销售额。

In [ ]:
# 按月份和产品大类分组汇总
category_monthly = df.groupby(['月份', '产品大类'])['销售金额'].sum().reset_index()

print('=== 产品大类每月销售数据透视表 ===')
pivot_table = category_monthly.pivot(index='月份', columns='产品大类', values='销售金额')
pivot_table.index = [f'{m}月' for m in pivot_table.index]
display(pivot_table)

# 绘制分组柱状图
categories = sorted(category_monthly['产品大类'].unique())
months = list(range(1, 13))
bar_width = 0.35
x_positions = list(months)

fig, ax = plt.subplots(figsize=(14, 7))

for i, cat in enumerate(categories):
    data = category_monthly[category_monthly['产品大类'] == cat]
    monthly_values = []
    for m in months:
        val = data[data['月份'] == m]['销售金额'].values
        monthly_values.append(val[0] if len(val) > 0 else 0)
    offset = (i - (len(categories) - 1) / 2) * bar_width
    bars = ax.bar([xi + offset for xi in x_positions], monthly_values, bar_width, label=cat)

    # 在柱子上标注数值
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{height:,.0f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords='offset points',
                       ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xlabel('月份', fontsize=12)
ax.set_ylabel('销售金额（元）', fontsize=12)
ax.set_title('各产品大类每月销售金额对比', fontsize=16, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels([f'{m}月' for m in months])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('category_monthly_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存为 category_monthly_comparison.png')

## 6. 全年各商品小类销售额对比

从分组汇总结果中提取 **产品小类** 作为 X 轴，**销售金额** 作为 Y 轴，使用 `bar()` 绘制柱状图进行对比分析。

In [ ]:
# 按产品小类分组汇总全年销售额
subcategory_sales = df.groupby('产品小类')['销售金额'].sum().sort_values(ascending=True).reset_index()
subcategory_sales.columns = ['产品小类', '销售总额']

print('=== 各产品小类全年销售总额 ===')
display(subcategory_sales)

# 绘制横向柱状图
fig, ax = plt.subplots(figsize=(12, 8))

# 使用渐变色
import numpy as np
n_cats = len(subcategory_sales)
colors = plt.cm.RdYlGn([i / (n_cats - 1) for i in range(n_cats)])

bars = ax.barh(subcategory_sales['产品小类'], subcategory_sales['销售总额'],
               color=colors, edgecolor='black', linewidth=0.5)

# 在柱子上标注数值
for bar in bars:
    width = bar.get_width()
    ax.annotate(f'  {width:,.0f} 元',
               xy=(width, bar.get_y() + bar.get_height() / 2),
               xytext=(5, 0), textcoords='offset points',
               ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('销售总额（元）', fontsize=12)
ax.set_ylabel('产品小类', fontsize=12)
ax.set_title('全年各产品小类销售额对比', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('subcategory_total_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存为 subcategory_total_sales.png')

# 占比分析
print('\n=== 各产品小类销售额占比（降序）===')
subcategory_with_pct = subcategory_sales.copy()
subcategory_with_pct['占比(%)'] = (subcategory_with_pct['销售总额'] / subcategory_with_pct['销售总额'].sum() * 100).round(2)
subcategory_with_pct = subcategory_with_pct.sort_values('销售总额', ascending=False)
subcategory_with_pct['累计占比(%)'] = subcategory_with_pct['占比(%)'].cumsum().round(2)
display(subcategory_with_pct)

## 7. 分析结论与建议

### 📊 数据概览
- 本次分析覆盖了 **2022年全年（1-12月）** 的销售数据
- 共包含 **40款产品**，**2,055条** 销售记录
- 产品分为 **2个大类**（运动套装、运动配件）和多个小类（上衣、短裤、长裤、袜子、发带、护腕等）

### 🔍 主要发现

1. **月度销售趋势**：各产品的月度销售额呈现一定波动性，部分月份销售额明显增长，可能与促销活动和季节性需求有关。

2. **品类销售差异**：
   - 运动套装类产品（上衣、短裤、长裤）因其单价较高，在总销售额中占比较大
   - 运动配件类产品（袜子、发带、护腕）单价低但销量大，是重要的流量入口

3. **产品集中度**：销售额排名前10的产品贡献了较大比例的销售额，体现了"爆款"效应。

4. **季节性特征**：不同品类的销售表现出明显的季节性特征，如运动套装在夏季销售更高。

### 💡 运营建议

1. **优化产品结构**：针对热销品类加大投入，同时对低销量产品进行优化或淘汰
2. **差异化营销**：根据不同品类的季节性波动特点，制定差异化的促销策略
3. **库存管理**：基于月度销售趋势预测，合理安排库存，降低库存成本
4. **产品组合**：将高单价产品与高流量配件进行组合销售，提升客单价

### 🤖 关于AI与个人发展

在完成本项目的过程中，我使用了AI编程助手（Claude Code）来辅助完成数据分析工作。这让我深刻体会到：

- **AI是效率倍增器**：在数据读取、代码编写、图表绘制等环节，AI能够快速生成高质量的代码，大幅提升工作效率
- **核心能力不可替代**：虽然AI可以辅助编写代码，但对业务逻辑的理解、数据分析思路的设计、结论的提炼等仍需人类思考
- **持续学习的重要性**：面对AI技术的快速发展，我们应当学会利用AI工具增强自身能力，同时保持对底层原理的学习和理解
- **未来展望**：在AI时代，核心竞争力将从"会写代码"转变为"能定义问题、设计解决方案、并有效利用AI工具实现目标"的复合能力